# OrigenLM — fine-tuning QLoRA (Kaggle T4)

Adapta Qwen2.5-Coder-7B-Instruct a los diálogos de OrigenLM.

- **Settings → Accelerator: GPU T4 x2**, Internet **ON**.
- **Add-ons → Secrets**: crea `HF_TOKEN` (token de Hugging Face con permiso Write) para subir el adaptador al final.
- El repo se baja como tarball a `/tmp` (GitHub ya es público).
- El adaptador queda en `/kaggle/working/qlora_origen` (persistente) y se sube a Hugging Face en la penúltima celda.

In [ ]:
%%bash
set -e
REPO="J3lk5d0x3l/OrigenLM"
REF="main"
rm -rf /tmp/origen /tmp/origen.tar.gz
curl -fsSL -o /tmp/origen.tar.gz "https://codeload.github.com/${REPO}/tar.gz/${REF}"
mkdir -p /tmp/origen
tar -xzf /tmp/origen.tar.gz -C /tmp/origen --strip-components=1
ls /tmp/origen


In [ ]:
%cd /tmp/origen
!pip -q install -r requirements.txt
# Kaggle puede traer torchao 0.10.0; PEFT actual requiere >0.16.0 y falla al inyectar LoRA.
# No necesitamos torchao: QLoRA aquí usa bitsandbytes. Quitar tras instalar dependencias.
!python -m pip uninstall -y torchao
!python scripts/preparar_sft.py
!python train_qlora.py --dry-run
!head -2 dataset/sft/train.jsonl

In [ ]:
%cd /tmp/origen
# Una época sobre los ejemplos actuales: minutos en T4. Sube --epochs para repetir.
!python train_qlora.py \
  --epochs 1 \
  --batch 2 --grad-acum 8 \
  --max-len 512 \
  --eval-cada 50 \
  --salida /kaggle/working/qlora_origen

In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base = "Qwen/Qwen2.5-Coder-7B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
# 4 bits como en el entrenamiento: evita el despachador de torchao (versión vieja en Kaggle)
cuant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = AutoModelForCausalLM.from_pretrained(base, quantization_config=cuant, device_map={"": 0})
modelo = PeftModel.from_pretrained(modelo, "/kaggle/working/qlora_origen/adapter")
modelo.eval()

mensajes = [{"role": "system", "content": "Eres Origen, un asistente claro y útil. Respondés en español."},
            {"role": "user", "content": "Hola, ¿quién eres?"}]
enc = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt")
ids = (enc["input_ids"] if hasattr(enc, "keys") else enc).to(modelo.device)
salida = modelo.generate(ids, max_new_tokens=120, temperature=0.3, do_sample=True,
                         pad_token_id=tok.eos_token_id)
print(tok.decode(salida[0][ids.shape[1]:], skip_special_tokens=True))
print(json.load(open("/kaggle/working/qlora_origen/meta.json")))

## Guardar el adaptador en Hugging Face

Sube `/kaggle/working/qlora_origen` a un repo **privado** `<tu-usuario>/origenlm-qlora`. Así el adaptador no depende de que Kaggle conserve la sesión.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Add-ons → Secrets → HF_TOKEN (token de Hugging Face con permiso Write)
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python /tmp/origen/scripts/subir_hf.py --repo origenlm-qlora --carpeta /kaggle/working/qlora_origen

## Después

Para traer el adaptador al codespace o a tu PC:

```bash
export HF_TOKEN=hf_...
python scripts/subir_hf.py --bajar --repo origenlm-qlora --carpeta checkpoints/qlora_origen
```

Para usarlo en el codespace (CPU) conviene fusionarlo y exportarlo a GGUF:

```bash
python train_qlora.py --fusionar        # deja checkpoints/qlora_origen/modelo_fusionado
python llama.cpp/convert_hf_to_gguf.py checkpoints/qlora_origen/modelo_fusionado --outfile models/origen-fusionado.gguf
```

Con pocos ejemplos el resultado aprende estilo, no conocimiento. Para mejorar de
verdad hay que sumar datos: ver `dataset/entradas/LEEME.md`. Guía completa: `docs/KAGGLE.md`.